In [ ]:

import re
from collections import defaultdict
from pathlib import Path
import pandas as pd


SERIES_MATRIX = Path("/content/GSE76427_series_matrix.txt")
assert SERIES_MATRIX.exists(), f"File not found: {SERIES_MATRIX}"

headers = {}
with open(SERIES_MATRIX) as f:
    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            break
        if not line.startswith("!Sample_"):
            continue
        parts = line.rstrip("\n").split("\t")
        key = parts[0]
        vals = [v.strip('"') for v in parts[1:]]
        if key not in headers:
            headers[key] = vals
        else:

            if isinstance(headers[key][0], str):
                headers[key] = [headers[key], vals]
            else:
                headers[key].append(vals)

geo    = headers["!Sample_geo_accession"]
titles = headers["!Sample_title"]
n      = len(geo)
assert n == 167, f"Expected 167 samples, got {n}"

chars = headers["!Sample_characteristics_ch1"]
char_lines = chars if isinstance(chars[0], list) else [chars]

per_sample = [defaultdict(str) for _ in range(n)]
for cline in char_lines:
    for i, v in enumerate(cline):
        if ":" in v:
            k, val = v.split(":", 1)
            per_sample[i][k.strip()] = val.strip()


rows = []
for i in range(n):
    d     = per_sample[i]
    title = titles[i]
    tissue_raw = d.get("tissue", "")

    if "primary" in tissue_raw.lower() or (
        "tumor" in title.lower()
        and "non-tumor" not in title.lower()
        and "adjacent" not in title.lower()
    ):
        tissue = "tumour"
    elif "non-tumor" in tissue_raw.lower() or "adjacent" in title.lower():
        tissue = "adjacent"
    else:
        raise ValueError(f"Cannot classify tissue for {geo[i]}: {title!r} / {tissue_raw!r}")

    patient = d.get("patient id", "")
    if not patient:
        m = re.search(r",\s*(\d+)", title)
        patient = f"HCC{m.group(1)}" if m else ""

    os_time  = d.get("duryears_os", "")
    os_event = d.get("event_os", "")

    rows.append({
        "sample":   geo[i],
        "patient":  patient,
        "tissue":   tissue,
        "OS_time":  os_time,
        "OS_event": os_event,
    })

df = pd.DataFrame(rows)


assert not df["sample"].duplicated().any(), "Duplicate sample IDs"
assert not df.duplicated(["patient", "tissue"]).any(), "Duplicate patient+tissue"
assert set(df["tissue"]) <= {"tumour", "adjacent"}
assert (df["patient"].str.strip() != "").all()
assert (df.loc[df.tissue == "tumour", "OS_time"]  != "").all()
assert (df.loc[df.tissue == "tumour", "OS_event"] != "").all()

print("Tissue counts:")
print(df["tissue"].value_counts())
print("\nTumour OS events:")
print(df.loc[df.tissue == "tumour", "OS_event"].value_counts())
print(f"\nUnique patients: {df['patient'].nunique()}")
print(f"Shape: {df.shape}")


OUT = Path("/content/external_metadata_reviewed.csv")
df.to_csv(OUT, index=False)
print(f"\nWrote → {OUT}")
print(df.head(8).to_string(index=False))

Tissue counts:
tissue
tumour      115
adjacent     52
Name: count, dtype: int64

Tumour OS events:
OS_event
0    92
1    23
Name: count, dtype: int64

Unique patients: 115
Shape: (167, 5)

Wrote → /content/external_metadata_reviewed.csv
    sample patient tissue OS_time OS_event
GSM2011285   HCC19 tumour    2.56        0
GSM2011286   HCC62 tumour    2.17        0
GSM2011287   HCC21 tumour    0.04        0
GSM2011288   HCC63 tumour    0.49        1
GSM2011289   HCC46 tumour    0.98        1
GSM2011290   HCC60 tumour    0.03        0
GSM2011291    HCC8 tumour    0.58        0
GSM2011292   HCC66 tumour    0.04        0
